<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week5_instructor_life_expectancy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 – Instructor demo: life expectancy vs. income, live

**For live use in the workshop, not a student handout.** Reproduces the slide's own life-expectancy-vs-GDP case study (level-level, quadratic, log-level, quadratic-on-logs) as real, re-runnable code.

**A genuine limitation, stated up front:** the hosted data (`worldbank_lifeexp_2017.csv`) is a single year (2017), 182 countries, three columns – there's no "try a different year" lever here the way there was for Week 4's hotel data. What this notebook *does* give you live flexibility on: re-visualising the four specifications together, and – tying directly into this week's "extreme values and influence" section – excluding specific countries live to show how much a handful of very high-GDP outliers are actually driving the fit.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

data_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/worldbank_lifeexp_2017.csv"
life_df = pd.read_csv(data_path)
life_df['ln_gdp_per_capita'] = np.log(life_df['gdp_per_capita'])
life_df.head()

## The four specifications, side by side

In [ ]:
def fit_all(df):
    return {
        'Level-level': smf.ols('life_exp ~ gdp_per_capita', data=df).fit(),
        'Quadratic (raw GDP)': smf.ols('life_exp ~ gdp_per_capita + I(gdp_per_capita**2)', data=df).fit(),
        'Log-level': smf.ols('life_exp ~ ln_gdp_per_capita', data=df).fit(),
        'Quadratic (log GDP)': smf.ols('life_exp ~ ln_gdp_per_capita + I(ln_gdp_per_capita**2)', data=df).fit(),
    }

models = fit_all(life_df)
for name, m in models.items():
    print(f'{name:22s} R2 = {m.rsquared:.3f}')

**Talking point:** log-level is already close to the quadratic-on-logs (R² barely moves, ~0.680 vs. ~0.682) – the extra curvature isn't buying much. The level-level R² (~0.44) looks much worse mainly because a handful of very rich countries are wildly out of scale on the raw-GDP axis, not because income and life expectancy are weakly related.

## Visualise the fits together

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

order = life_df.sort_values('gdp_per_capita')
axes[0].scatter(life_df['gdp_per_capita'], life_df['life_exp'], alpha=0.5, color='#1a1a2e')
axes[0].plot(order['gdp_per_capita'], models['Level-level'].predict(order), color='#e63946', label='Level-level')
axes[0].plot(order['gdp_per_capita'], models['Quadratic (raw GDP)'].predict(order), color='#457b9d', label='Quadratic')
axes[0].set_xlabel('GDP per capita ($1,000s)'); axes[0].set_ylabel('Life expectancy')
axes[0].set_title('Raw GDP scale'); axes[0].legend()

order_log = life_df.sort_values('ln_gdp_per_capita')
axes[1].scatter(life_df['ln_gdp_per_capita'], life_df['life_exp'], alpha=0.5, color='#1a1a2e')
axes[1].plot(order_log['ln_gdp_per_capita'], models['Log-level'].predict(order_log), color='#e63946', label='Log-level')
axes[1].plot(order_log['ln_gdp_per_capita'], models['Quadratic (log GDP)'].predict(order_log), color='#457b9d', label='Quadratic on logs')
axes[1].set_xlabel('ln(GDP per capita)'); axes[1].set_ylabel('Life expectancy')
axes[1].set_title('Log GDP scale'); axes[1].legend()

plt.tight_layout()
plt.show()

## Live: how much are a few outliers driving the level-level fit?

In [ ]:
life_df.sort_values('gdp_per_capita', ascending=False).head(5)[['country', 'gdp_per_capita', 'life_exp']]

**Change the country list below and re-run** to show live how much the level-level fit moves when you drop the highest-GDP countries – a direct, concrete version of this week's "extreme values and influence" point.

In [ ]:
EXCLUDE = ['Qatar', 'Macao SAR, China', 'Luxembourg']  # <-- edit this list and re-run

life_excl = life_df[~life_df['country'].isin(EXCLUDE)]
m_level_excl = smf.ols('life_exp ~ gdp_per_capita', data=life_excl).fit()

print(f"Level-level R2, all {len(life_df)} countries:      {models['Level-level'].rsquared:.3f}")
print(f"Level-level R2, excluding {len(EXCLUDE)} countries: {m_level_excl.rsquared:.3f}")
print()
print('Slope, all countries:      ', round(models['Level-level'].params['gdp_per_capita'], 3))
print('Slope, excluding outliers: ', round(m_level_excl.params['gdp_per_capita'], 3))

## Live: predict for a hypothetical GDP per capita

In [ ]:
GDP_VALUE = 20.0  # <-- GDP per capita in $1,000s, edit and re-run

new_row = pd.DataFrame({'gdp_per_capita': [GDP_VALUE], 'ln_gdp_per_capita': [np.log(GDP_VALUE)]})
for name, m in models.items():
    pred = m.predict(new_row).iloc[0]
    print(f'{name:22s} predicts {pred:.1f} years')

**Talking point:** watch how far apart the level-level and log-level predictions can get away from the middle of the data – a live illustration of why functional form matters for anything beyond description.